# Catalog, DssPath & delete

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # use the in-place build of the repo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyheclib as phl

plt.style.use('ggplot')
# restart the kernel if this is not the version / location you expect
print('pyheclib', phl.__version__, 'from', os.path.dirname(phl.__file__))

if os.path.exists('catalog.dss'):
    os.remove('catalog.dss')
dss_file = phl.open('catalog.dss')
dss_file

pyheclib 0.4.0 from /Users/nzahasan/Development/pyheclib/pyheclib


<DssFile open 'catalog.dss'>

In [2]:
hours = pd.date_range('2020-01-01 01:00', '2020-04-01 00:00', freq='h')
series = pd.Series(np.arange(hours.size, dtype=float), index=hours)
for basin in ('NORTH', 'SOUTH'):
    dss_file.write(phl.RegularTs.from_series(f'/{basin}/GAGE/FLOW//1HOUR/OBS/', series, unit='CFS'))
    dss_file.write(phl.RegularTs.from_series(f'/{basin}/GAGE/FLOW//1HOUR/SIM/', series, unit='CFS'))
    dss_file.write(phl.PairedData(f'/{basin}/GAGE/STAGE-FLOW///RATING/', [0, 1, 2.0], [0, 5, 20.0]))
dss_file.write(phl.GriddedData('/SHG/NORTH/PRECIP/01JAN2020:0000/01JAN2020:0100/RADAR/', np.ones((3, 3)), 2000.0, 0.0, 0.0))

## Catalog
One path per stored record, time series are stored in dated blocks.

In [3]:
for path, code in dss_file.catalog(with_types=True):
    print(f'{code:4d}  {path}')

 105  /NORTH/GAGE/FLOW/01Jan2020/1Hour/OBS/
 105  /NORTH/GAGE/FLOW/01Feb2020/1Hour/OBS/
 105  /NORTH/GAGE/FLOW/01Mar2020/1Hour/OBS/
 105  /NORTH/GAGE/FLOW/01Jan2020/1Hour/SIM/
 105  /NORTH/GAGE/FLOW/01Feb2020/1Hour/SIM/
 105  /NORTH/GAGE/FLOW/01Mar2020/1Hour/SIM/
 205  /NORTH/GAGE/STAGE-FLOW///RATING/
 105  /SOUTH/GAGE/FLOW/01Jan2020/1Hour/OBS/
 105  /SOUTH/GAGE/FLOW/01Feb2020/1Hour/OBS/
 105  /SOUTH/GAGE/FLOW/01Mar2020/1Hour/OBS/
 105  /SOUTH/GAGE/FLOW/01Jan2020/1Hour/SIM/
 105  /SOUTH/GAGE/FLOW/01Feb2020/1Hour/SIM/
 105  /SOUTH/GAGE/FLOW/01Mar2020/1Hour/SIM/
 205  /SOUTH/GAGE/STAGE-FLOW///RATING/
 430  /SHG/NORTH/PRECIP/01JAN2020:0000/01JAN2020:0100/RADAR/


The condensed catalog combines the blocks of each time series into one path.

In [4]:
condensed = dss_file.catalog(condensed=True)
for path in condensed:
    print(path)
assert len(condensed) == 7

/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/OBS/
/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/SIM/
/NORTH/GAGE/STAGE-FLOW///RATING/
/SOUTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/OBS/
/SOUTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/SIM/
/SOUTH/GAGE/STAGE-FLOW///RATING/
/SHG/NORTH/PRECIP/01JAN2020:0000/01JAN2020:0100/RADAR/


In [5]:
dss_file.catalog('/NORTH/*/FLOW/*/*/*/', condensed=True)

[DssPath('/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/OBS/'),
 DssPath('/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/SIM/')]

## DssPath

In [6]:
path = condensed[0]
print(repr(path))
print(path.A, path.C, path.D)
print(path.without_date())
print(path.replace(F='FORECAST'))
print(phl.DssPath.from_parts(A='NORTH', C='STAGE-FLOW', F='RATING'))
assert path == str(path).lower()     # comparison is case insensitive

# a condensed path reads the whole series
assert dss_file.read(path).values.size == hours.size

DssPath('/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/OBS/')
NORTH FLOW 01JAN2020-01MAR2020
/NORTH/GAGE/FLOW//1Hour/OBS/
/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/FORECAST/
/NORTH//STAGE-FLOW///RATING/


## find & delete with * wild characters
`*` can be used at the start or end of any part.

In [7]:
dss_file.find('/*/*/FLOW/*/*/SIM/')

[DssPath('/NORTH/GAGE/FLOW/01Jan2020/1Hour/SIM/'),
 DssPath('/NORTH/GAGE/FLOW/01Feb2020/1Hour/SIM/'),
 DssPath('/NORTH/GAGE/FLOW/01Mar2020/1Hour/SIM/'),
 DssPath('/SOUTH/GAGE/FLOW/01Jan2020/1Hour/SIM/'),
 DssPath('/SOUTH/GAGE/FLOW/01Feb2020/1Hour/SIM/'),
 DssPath('/SOUTH/GAGE/FLOW/01Mar2020/1Hour/SIM/')]

In [8]:
deleted = dss_file.delete('/SOUTH/*/*/*/*/*/')         # everything in basin SOUTH
print(len(deleted), 'records deleted')
assert dss_file.find('/SOUTH/*/*/*/*/*/') == []

dss_file.delete('/NORTH/GAGE/FLOW/01JAN2020-01FEB2020/1HOUR/SIM/')   # blocks within a date range
dss_file.delete('/*/*/*/*/*/RAD*/')                                  # partial part match
dss_file.catalog(condensed=True)

7 records deleted


[DssPath('/NORTH/GAGE/FLOW/01JAN2020-01MAR2020/1Hour/OBS/'),
 DssPath('/NORTH/GAGE/FLOW/01MAR2020/1Hour/SIM/'),
 DssPath('/NORTH/GAGE/STAGE-FLOW///RATING/')]

In [9]:
dss_file.close()
print('all checks passed')

all checks passed
